In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class Perceptron:
    def __init__(self, learning_rate: float = 0.1, epochs: int = 100, shuffle: bool = True):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.shuffle = shuffle
        self.weights = None
        self.bias = None
        self.errors_history_ = []
        self.epochs_run_ = 0

    def activation(self, z):
        return 1 if z >= 0 else 0

    def fit(self, X: np.ndarray, y: np.ndarray):
        n_samples, n_features = X.shape
        self.weights = np.random.uniform(-0.01, 0.01, size=n_features)
        self.bias = float(np.random.uniform(-0.01, 0.01))
        self.errors_history_.clear()

        for epoch in range(1, self.epochs + 1):
            idx = np.random.permutation(n_samples) if self.shuffle else np.arange(n_samples)
            errors = 0
            for i in idx:
                xi = X[i]
                zi = float(np.dot(xi, self.weights) + self.bias)
                y_hat = self.activation(zi)
                error = y[i] - y_hat
                if error != 0:
                    self.weights = self.weights + self.learning_rate * error * xi
                    self.bias = self.bias + self.learning_rate * error
                    errors += 1
            self.errors_history_.append(errors)
            if errors == 0:
                self.epochs_run_ = epoch
                break
        else:
            self.epochs_run_ = self.epochs
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        z = X @ self.weights + self.bias
        return (z >= 0).astype(int)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        return (self.predict(X) == y).mean()


def save_and_load_csv(df: pd.DataFrame, path: str) -> pd.DataFrame:
    df.to_csv(path, index=False)
    return pd.read_csv(path)

def make_and_dataset_csv(path: str) -> pd.DataFrame:
    X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
    y = np.array([0, 0, 0, 1], dtype=int)
    df = pd.DataFrame({'x1': X[:,0], 'x2': X[:,1], 'y': y})
    return save_and_load_csv(df, path)

def make_gaussians_separable_csv(path: str, n_per_class: int = 120, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    mean0 = np.array([-2.0, -2.0])
    mean1 = np.array([ 2.0,  2.0])
    cov = np.array([[0.7, 0.15],[0.15, 0.7]])
    X0 = rng.multivariate_normal(mean0, cov, size=n_per_class)
    X1 = rng.multivariate_normal(mean1, cov, size=n_per_class)
    y0 = np.zeros(n_per_class, dtype=int)
    y1 = np.ones(n_per_class, dtype=int)
    X = np.vstack([X0, X1])
    y = np.concatenate([y0, y1])
    df = pd.DataFrame({'x1': X[:,0], 'x2': X[:,1], 'y': y})
    return save_and_load_csv(df, path)

def plot_decision_boundary(X: np.ndarray, y: np.ndarray, model: Perceptron, title: str, filename: str):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,5))
    plt.scatter(X[y==0,0], X[y==0,1], c='tab:blue', label='Clase 0', s=25)
    plt.scatter(X[y==1,0], X[y==1,1], c='tab:orange', label='Clase 1', s=25)
    w, b = model.weights, model.bias
    if abs(w[1]) > 1e-10:
        x_vals = np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200)
        y_vals = -(w[0]*x_vals + b)/w[1]
        plt.plot(x_vals, y_vals, 'k--', lw=2, label='Frontera de decisión')
    else:
        x_const = -b/(w[0] + 1e-12)
        plt.axvline(x_const, color='k', linestyle='--', lw=2, label='Frontera de decisión')
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

def plot_errors(model: Perceptron, title: str, filename: str):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.plot(range(1, len(model.errors_history_)+1), model.errors_history_, marker='o')
    plt.title(title)
    plt.xlabel('Época')
    plt.ylabel('Errores (actualizaciones)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()


and_df = make_and_dataset_csv('and_dataset.csv')
X_and = and_df[['x1','x2']].to_numpy(float)
y_and = and_df['y'].to_numpy(int)

per_and = Perceptron(learning_rate=0.2, epochs=100)
per_and.fit(X_and, y_and)
acc_and = per_and.score(X_and, y_and)
plot_decision_boundary(X_and, y_and, per_and, f'AND: frontera (acc={acc_and:.2f}, épocas={per_and.epochs_run_})', 'and_boundary.png')
plot_errors(per_and, 'AND: errores por época', 'and_errors.png')

gauss_df = make_gaussians_separable_csv('gauss_dataset.csv', n_per_class=120, seed=3)
X_g = gauss_df[['x1','x2']].to_numpy(float)
y_g = gauss_df['y'].to_numpy(int)

per_g = Perceptron(learning_rate=0.05, epochs=100)
per_g.fit(X_g, y_g)
acc_g = per_g.score(X_g, y_g)
plot_decision_boundary(X_g, y_g, per_g, f'Gauss separable: frontera (acc={acc_g:.2f}, épocas={per_g.epochs_run_})', 'gauss_boundary.png')
plot_errors(per_g, 'Gauss separable: errores por época', 'gauss_errors.png')

print('=== REPORTE ===')
print('--- AND ---')
print('Pesos:', per_and.weights)
print('Sesgo:', round(per_and.bias, 4))
print('Épocas usadas:', per_and.epochs_run_)
print('Exactitud:', round(acc_and, 3))
print('Archivos: and_dataset.csv, and_boundary.png, and_errors.png')
print('')
print('--- GAUSS ---')
print('Pesos:', per_g.weights)
print('Sesgo:', round(per_g.bias, 4))
print('Épocas usadas:', per_g.epochs_run_)
print('Exactitud:', round(acc_g, 3))
print('Archivos: gauss_dataset.csv, gauss_boundary.png, gauss_errors.png')

=== REPORTE ===
--- AND ---
Pesos: [0.20546335 0.20832825]
Sesgo: -0.4014
Épocas usadas: 5
Exactitud: 1.0
Archivos: and_dataset.csv, and_boundary.png, and_errors.png

--- GAUSS ---
Pesos: [0.14694467 0.10382937]
Sesgo: -0.0537
Épocas usadas: 2
Exactitud: 1.0
Archivos: gauss_dataset.csv, gauss_boundary.png, gauss_errors.png
